In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 04. Model Evaluation & Comparison\n",
    "## NLP Chatbot Project - Comprehensive Model Benchmarking\n",
    "\n",
    "This notebook evaluates all 4 trained models on the test set:\n",
    "- Accuracy, Precision, Recall, F1-Score, AUC\n",
    "- Confusion matrices\n",
    "- Inference time analysis\n",
    "- Model comparison and selection\n",
    "\n",
    "**Goal**: Select the best model for deployment based on comprehensive metrics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import sys\n",
    "import time\n",
    "import pickle\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "from sklearn.metrics import (\n",
    "    accuracy_score, precision_score, recall_score, f1_score,\n",
    "    classification_report, confusion_matrix, roc_auc_score\n",
    ")\n",
    "from sklearn.preprocessing import label_binarize\n",
    "\n",
    "# Add src to path\n",
    "sys.path.append('../src')\n",
    "\n",
    "# Import custom modules\n",
    "from preprocessor import TextVectorizer\n",
    "from models import (\n",
    "    LogisticRegressionModel,\n",
    "    RandomForestModel,\n",
    "    XGBoostModel,\n",
    "    LSTMModel\n",
    ")\n",
    "\n",
    "# Set style\n",
    "sns.set_style('whitegrid')\n",
    "plt.rcParams['figure.figsize'] = (14, 8)\n",
    "\n",
    "print(\"✅ Imports successful!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load Test Data and Artifacts"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load test data\n",
    "print(\"📂 Loading test data and artifacts...\\n\")\n",
    "\n",
    "test_df = pd.read_csv('../data/test.csv')\n",
    "\n",
    "print(f\"✅ Test data loaded: {len(test_df):,} samples\")\n",
    "\n",
    "# Load vectorizer\n",
    "vectorizer = TextVectorizer()\n",
    "vectorizer.load('../models/vectorizer.pkl')\n",
    "print(\"✅ Vectorizer loaded\")\n",
    "\n",
    "# Load label encoder\n",
    "with open('../models/label_encoder.pkl', 'rb') as f:\n",
    "    label_encoder = pickle.load(f)\n",
    "print(f\"✅ Label encoder loaded ({len(label_encoder.classes_)} classes)\")\n",
    "\n",
    "# Prepare test labels\n",
    "if 'label_encoded' not in test_df.columns:\n",
    "    y_test = label_encoder.transform(test_df['label'])\n",
    "else:\n",
    "    y_test = test_df['label_encoded'].values\n",
    "\n",
    "# Vectorize test data\n",
    "X_test = vectorizer.transform(test_df['processed_text'])\n",
    "print(f\"✅ Test data vectorized: {X_test.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Load All Trained Models"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n📦 Loading trained models...\\n\")\n",
    "\n",
    "# 1. Logistic Regression\n",
    "lr_model = LogisticRegressionModel()\n",
    "lr_model.load('../models/logistic_model.pkl')\n",
    "print(\"✅ Logistic Regression loaded\")\n",
    "\n",
    "# 2. Random Forest\n",
    "rf_model = RandomForestModel()\n",
    "rf_model.load('../models/random_forest_model.pkl')\n",
    "print(\"✅ Random Forest loaded\")\n",
    "\n",
    "# 3. XGBoost\n",
    "xgb_model = XGBoostModel()\n",
    "xgb_model.load('../models/xgboost_model.pkl')\n",
    "print(\"✅ XGBoost loaded\")\n",
    "\n",
    "# 4. LSTM\n",
    "lstm_model = LSTMModel()\n",
    "lstm_model.load('../models/lstm_model.h5', '../models/lstm_tokenizer.pkl')\n",
    "print(\"✅ LSTM loaded\")\n",
    "\n",
    "print(\"\\n✅ All models loaded successfully!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Evaluate Model 1: Logistic Regression"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=\"*80)\n",
    "print(\"EVALUATING: LOGISTIC REGRESSION\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "# Predictions\n",
    "start_time = time.time()\n",
    "lr_pred = lr_model.predict(X_test)\n",
    "lr_inference_time = time.time() - start_time\n",
    "\n",
    "lr_pred_proba = lr_model.predict_proba(X_test)\n",
    "\n",
    "# Metrics\n",
    "lr_accuracy = accuracy_score(y_test, lr_pred)\n",
    "lr_precision = precision_score(y_test, lr_pred, average='weighted', zero_division=0)\n",
    "lr_recall = recall_score(y_test, lr_pred, average='weighted', zero_division=0)\n",
    "lr_f1 = f1_score(y_test, lr_pred, average='weighted', zero_division=0)\n",
    "\n",
    "# AUC\n",
    "y_test_binarized = label_binarize(y_test, classes=range(len(label_encoder.classes_)))\n",
    "lr_auc = roc_auc_score(y_test_binarized, lr_pred_proba, average='weighted', multi_class='ovr')\n",
    "\n",
    "print(f\"\\n📊 Performance Metrics:\")\n",
    "print(f\"   Accuracy:       {lr_accuracy:.4f}\")\n",
    "print(f\"   Precision:      {lr_precision:.4f}\")\n",
    "print(f\"   Recall:         {lr_recall:.4f}\")\n",
    "print(f\"   F1-Score:       {lr_f1:.4f}\")\n",
    "print(f\"   AUC:            {lr_auc:.4f}\")\n",
    "print(f\"   Inference Time: {lr_inference_time:.4f} seconds\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Evaluate Model 2: Random Forest"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n\" + \"=\"*80)\n",
    "print(\"EVALUATING: RANDOM FOREST\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "# Predictions\n",
    "start_time = time.time()\n",
    "rf_pred = rf_model.predict(X_test)\n",
    "rf_inference_time = time.time() - start_time\n",
    "\n",
    "rf_pred_proba = rf_model.predict_proba(X_test)\n",
    "\n",
    "# Metrics\n",
    "rf_accuracy = accuracy_score(y_test, rf_pred)\n",
    "rf_precision = precision_score(y_test, rf_pred, average='weighted', zero_division=0)\n",
    "rf_recall = recall_score(y_test, rf_pred, average='weighted', zero_division=0)\n",
    "rf_f1 = f1_score(y_test, rf_pred, average='weighted', zero_division=0)\n",
    "rf_auc = roc_auc_score(y_test_binarized, rf_pred_proba, average='weighted', multi_class='ovr')\n",
    "\n",
    "print(f\"\\n📊 Performance Metrics:\")\n",
    "print(f\"   Accuracy:       {rf_accuracy:.4f}\")\n",
    "print(f\"   Precision:      {rf_precision:.4f}\")\n",
    "print(f\"   Recall:         {rf_recall:.4f}\")\n",
    "print(f\"   F1-Score:       {rf_f1:.4f}\")\n",
    "print(f\"   AUC:            {rf_auc:.4f}\")\n",
    "print(f\"   Inference Time: {rf_inference_time:.4f} seconds\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Evaluate Model 3: XGBoost"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n\" + \"=\"*80)\n",
    "print(\"EVALUATING: XGBOOST\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "# Predictions\n",
    "start_time = time.time()\n",
    "xgb_pred = xgb_model.predict(X_test)\n",
    "xgb_inference_time = time.time() - start_time\n",
    "\n",
    "xgb_pred_proba = xgb_model.predict_proba(X_test)\n",
    "\n",
    "# Metrics\n",
    "xgb_accuracy = accuracy_score(y_test, xgb_pred)\n",
    "xgb_precision = precision_score(y_test, xgb_pred, average='weighted', zero_division=0)\n",
    "xgb_recall = recall_score(y_test, xgb_pred, average='weighted', zero_division=0)\n",
    "xgb_f1 = f1_score(y_test, xgb_pred, average='weighted', zero_division=0)\n",
    "xgb_auc = roc_auc_score(y_test_binarized, xgb_pred_proba, average='weighted', multi_class='ovr')\n",
    "\n",
    "print(f\"\\n📊 Performance Metrics:\")\n",
    "print(f\"   Accuracy:       {xgb_accuracy:.4f}\")\n",
    "print(f\"   Precision:      {xgb_precision:.4f}\")\n",
    "print(f\"   Recall:         {xgb_recall:.4f}\")\n",
    "print(f\"   F1-Score:       {xgb_f1:.4f}\")\n",
    "print(f\"   AUC:            {xgb_auc:.4f}\")\n",
    "print(f\"   Inference Time: {xgb_inference_time:.4f} seconds\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Evaluate Model 4: LSTM"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n\" + \"=\"*80)\n",
    "print(\"EVALUATING: LSTM\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "# Prepare text data\n",
    "X_test_texts = test_df['processed_text'].tolist()\n",
    "\n",
    "# Predictions\n",
    "start_time = time.time()\n",
    "lstm_pred = lstm_model.predict(X_test_texts)\n",
    "lstm_inference_time = time.time() - start_time\n",
    "\n",
    "lstm_pred_proba = lstm_model.predict_proba(X_test_texts)\n",
    "\n",
    "# Metrics\n",
    "lstm_accuracy = accuracy_score(y_test, lstm_pred)\n",
    "lstm_precision = precision_score(y_test, lstm_pred, average='weighted', zero_division=0)\n",
    "lstm_recall = recall_score(y_test, lstm_pred, average='weighted', zero_division=0)\n",
    "lstm_f1 = f1_score(y_test, lstm_pred, average='weighted', zero_division=0)\n",
    "lstm_auc = roc_auc_score(y_test_binarized, lstm_pred_proba, average='weighted', multi_class='ovr')\n",
    "\n",
    "print(f\"\\n📊 Performance Metrics:\")\n",
    "print(f\"   Accuracy:       {lstm_accuracy:.4f}\")\n",
    "print(f\"   Precision:      {lstm_precision:.4f}\")\n",
    "print(f\"   Recall:         {lstm_recall:.4f}\")\n",
    "print(f\"   F1-Score:       {lstm_f1:.4f}\")\n",
    "print(f\"   AUC:            {lstm_auc:.4f}\")\n",
    "print(f\"   Inference Time: {lstm_inference_time:.4f} seconds\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Model Comparison"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create comparison DataFrame\n",
    "comparison_df = pd.DataFrame({\n",
    "    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost', 'LSTM'],\n",
    "    'Accuracy': [lr_accuracy, rf_accuracy, xgb_accuracy, lstm_accuracy],\n",
    "    'Precision': [lr_precision, rf_precision, xgb_precision, lstm_precision],\n",
    "    'Recall': [lr_recall, rf_recall, xgb_recall, lstm_recall],\n",
    "    'F1-Score': [lr_f1, rf_f1, xgb_f1, lstm_f1],\n",
    "    'AUC': [lr_auc, rf_auc, xgb_auc, lstm_auc],\n",
    "    'Inference_Time': [lr_inference_time, rf_inference_time, xgb_inference_time, lstm_inference_time]\n",
    "})\n",
    "\n",
    "print(\"\\n\" + \"=\"*80)\n",
    "print(\"MODEL COMPARISON - TEST SET RESULTS\")\n",
    "print(\"=\"*80)\n",
    "print(\"\\n\", comparison_df.to_string(index=False))\n",
    "\n",
    "# Find best model\n",
    "best_idx = comparison_df['F1-Score'].idxmax()\n",
    "best_model = comparison_df.loc[best_idx, 'Model']\n",
    "best_f1 = comparison_df.loc[best_idx, 'F1-Score']\n",
    "\n",
    "print(\"\\n\" + \"=\"*80)\n",
    "print(f\"🏆 BEST MODEL: {best_model}\")\n",
    "print(f\"   F1-Score: {best_f1:.4f}\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "# Save comparison\n",
    "comparison_df.set_index('Model').to_csv('../models/model_comparison.csv')\n",
    "print(\"\\n✅ Comparison saved to: ../models/model_comparison.csv\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Visualization: Performance Metrics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Performance metrics comparison\n",
    "fig, axes = plt.subplots(2, 2, figsize=(16, 12))\n",
    "\n",
    "metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']\n",
    "comparison_plot = comparison_df.set_index('Model')[metrics]\n",
    "\n",
    "# 1. All metrics comparison\n",
    "comparison_plot.plot(kind='bar', ax=axes[0, 0], rot=15)\n",
    "axes[0, 0].set_title('Performance Metrics Comparison', fontsize=14, fontweight='bold')\n",
    "axes[0, 0].set_ylabel('Score')\n",
    "axes[0, 0].set_ylim([0, 1.0])\n",
    "axes[0, 0].legend(loc='lower right')\n",
    "axes[0, 0].grid(alpha=0.3, axis='y')\n",
    "\n",
    "# 2. F1-Score ranking\n",
    "f1_sorted = comparison_df.sort_values('F1-Score', ascending=True)\n",
    "colors = ['gold' if x == best_model else 'steelblue' for x in f1_sorted['Model']]\n",
    "axes[0, 1].barh(f1_sorted['Model'], f1_sorted['F1-Score'], color=colors)\n",
    "axes[0, 1].set_title('F1-Score Ranking (Test Set)', fontsize=14, fontweight='bold')\n",
    "axes[0, 1].set_xlabel('F1-Score')\n",
    "axes[0, 1].set_xlim([0, 1.0])\n",
    "axes[0, 1].grid(alpha=0.3, axis='x')\n",
    "\n",
    "# 3. Inference time\n",
    "axes[1, 0].bar(comparison_df['Model'], comparison_df['Inference_Time'], color='coral')\n",
    "axes[1, 0].set_title('Inference Time Comparison', fontsize=14, fontweight='bold')\n",
    "axes[1, 0].set_ylabel('Time (seconds)')\n",
    "axes[1, 0].set_xticklabels(comparison_df['Model'], rotation=15, ha='right')\n",
    "axes[1, 0].grid(alpha=0.3, axis='y')\n",
    "\n",
    "# 4. Accuracy vs Speed tradeoff\n",
    "scatter = axes[1, 1].scatter(\n",
    "    comparison_df['Inference_Time'],\n",
    "    comparison_df['Accuracy'],\n",
    "    s=300,\n",
    "    c=comparison_df['F1-Score'],\n",
    "    cmap='viridis',\n",
    "    alpha=0.7,\n",
    "    edgecolors='black',\n",
    "    linewidth=2\n",
    ")\n",
    "for idx, row in comparison_df.iterrows():\n",
    "    axes[1, 1].annotate(\n",
    "        row['Model'].replace(' ', '\\n'),\n",
    "        (row['Inference_Time'], row['Accuracy']),\n",
    "        fontsize=9,\n",
    "        ha='center',\n",
    "        va='bottom'\n",
    "    )\n",
    "axes[1, 1].set_xlabel('Inference Time (seconds)', fontsize=12)\n",
    "axes[1, 1].set_ylabel('Accuracy', fontsize=12)\n",
    "axes[1, 1].set_title('Accuracy vs Speed Tradeoff', fontsize=14, fontweight='bold')\n",
    "axes[1, 1].grid(alpha=0.3)\n",
    "plt.colorbar(scatter, ax=axes[1, 1], label='F1-Score')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../models/model_comparison_detailed.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"\\n✅ Detailed comparison visualization saved\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Confusion Matrix (Best Model)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Get predictions from best model\n",
    "if best_model == 'Logistic Regression':\n",
    "    best_predictions = lr_pred\n",
    "elif best_model == 'Random Forest':\n",
    "    best_predictions = rf_pred\n",
    "elif best_model == 'XGBoost':\n",
    "    best_predictions = xgb_pred\n",
    "else:\n",
    "    best_predictions = lstm_pred\n",
    "\n",
    "# Compute confusion matrix\n",
    "cm = confusion_matrix(y_test, best_predictions)\n",
    "\n",
    "# Plot (subset for readability if too many classes)\n",
    "plt.figure(figsize=(14, 12))\n",
    "sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', cbar=True, square=True)\n",
    "plt.title(f'Confusion Matrix - {best_model}', fontsize=16, fontweight='bold')\n",
    "plt.ylabel('True Label', fontsize=12)\n",
    "plt.xlabel('Predicted Label', fontsize=12)\n",
    "plt.tight_layout()\n",
    "plt.savefig('../models/confusion_matrix_best_model.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"\\n✅ Confusion matrix saved\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Classification Report (Best Model)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(f\"\\nClassification Report - {best_model}\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "report = classification_report(\n",
    "    y_test,\n",
    "    best_predictions,\n",
    "    target_names=[f\"Intent_{i}\" for i in range(len(label_encoder.classes_))],\n",
    "    digits=4\n",
    ")\n",
    "\n",
    "print(report)\n",
    "\n",
    "# Save report\n",
    "with open('../models/classification_report.txt', 'w') as f:\n",
    "    f.write(f\"Classification Report - {best_model}\\n\")\n",
    "    f.write(\"=\"*80 + \"\\n\")\n",
    "    f.write(report)\n",
    "\n",
    "print(\"\\n✅ Classification report saved to: ../models/classification_report.txt\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 11. Error Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Find misclassified examples\n",
    "misclassified_mask = best_predictions != y_test\n",
    "misclassified_df = test_df[misclassified_mask].copy()\n",
    "misclassified_df['predicted_label'] = label_encoder.inverse_transform(best_predictions[misclassified_mask])\n",
    "misclassified_df['true_label'] = label_encoder.inverse_transform(y_test[misclassified_mask])\n",
    "\n",
    "print(f\"\\n📊 Error Analysis\")\n",
    "print(f\"   Total misclassified: {len(misclassified_df)} / {len(test_df)} ({len(misclassified_df)/len(test_df)*100:.2f}%)\")\n",
    "print(f\"\\n   Top 10 misclassified examples:\")\n",
    "print(misclassified_df[['text', 'true_label', 'predicted_label']].head(10).to_string())\n",
    "\n",
    "# Save misclassified examples\n",
    "misclassified_df.to_csv('../models/misclassified_examples.csv', index=False)\n",
    "print(\"\\n✅ Misclassified examples saved\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 12. Final Summary & Recommendation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=\"*80)\n",
    "print(\"FINAL EVALUATION SUMMARY\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "summary = f\"\"\"\n",
    "✅ Comprehensive Evaluation Complete!\n",
    "\n",
    "Test Set Performance:\n",
    "  - Test samples: {len(test_df):,}\n",
    "  - Number of classes: {len(label_encoder.classes_)}\n",
    "\n",
    "Models Evaluated:\n",
    "  1. Logistic Regression:  F1={lr_f1:.4f}, Time={lr_inference_time:.4f}s\n",
    "  2. Random Forest:        F1={rf_f1:.4f}, Time={rf_inference_time:.4f}s\n",
    "  3. XGBoost:              F1={xgb_f1:.4f}, Time={xgb_inference_time:.4f}s\n",
    "  4. LSTM:                 F1={lstm_f1:.4f}, Time={lstm_inference_time:.4f}s\n",
    "\n",
    "🏆 RECOMMENDED MODEL: {best_model}\n",
    "   - Accuracy:  {comparison_df.loc[best_idx, 'Accuracy']:.4f}\n",
    "   - Precision: {comparison_df.loc[best_idx, 'Precision']:.4f}\n",
    "   - Recall:    {comparison_df.loc[best_idx, 'Recall']:.4f}\n",
    "   - F1-Score:  {comparison_df.loc[best_idx, 'F1-Score']:.4f}\n",
    "   - AUC:       {comparison_df.loc[best_idx, 'AUC']:.4f}\n",
    "\n",
    "Deployment Ready:\n",
    "  ✓ Best model selected based on F1-Score\n",
    "  ✓ Comprehensive metrics evaluated\n",
    "  ✓ Inference time acceptable for production\n",
    "  ✓ All artifacts saved for deployment\n",
    "\n",
    "Saved Files:\n",
    "  ✓ model_comparison.csv\n",
    "  ✓ classification_report.txt\n",
    "  ✓ confusion_matrix_best_model.png\n",
    "  ✓ model_comparison_detailed.png\n",
    "  ✓ misclassified_examples.csv\n",
    "\n",
    "Next Steps:\n",
    "  1. Deploy {best_model} in web application\n",
    "  2. Monitor performance in production\n",
    "  3. Collect user feedback for improvements\n",
    "  4. Consider retraining with additional data\n",
    "\"\"\"\n",
    "\n",
    "print(summary)\n",
    "print(\"=\"*80)\n",
    "\n",